In [ ]:
import math
from collections import Counter, defaultdict

tinyNum = 1e-12  # just to avoid log(0)

def makeLower(text):
    return text.lower().strip()

def tokenize(sentence):
    # remove punctuations before splitting 
    token = makeLower(sentence).replace(".", " ").replace("!", " ").replace("?", " ").replace(";", " ").replace(":", " ").split()
    #print(["<s>"] + token + ["<e>"]) 
    return ["<s>"] + token + ["<e>"]

def bigram(ppSen):
    unknownCapacity = 0
    wordCounts = Counter()
    tokenizedSentence = []
    
    for line in ppSen:
        tokens = tokenize(line)
        tokenizedSentence.append(tokens)
        wordCounts.update(tokens) 

    vocab = set()  
    for w, c in wordCounts.items():
        # wordCounts.items() returns pairs: (word, count) so only add the word to the vocabulary
        # do not take into consideration OOV words 
        if w  in {"<s>", "<e>"} or c > unknownCapacity:
            vocab.add(w)

    vocab.add("<unk>")

    unigram = Counter()
    bigram = Counter()
    
    #calc the counts 
    for tokens in tokenizedSentence:
        updatedTokens = [w if w in vocab else "<unk>" for w in tokens]  # adding counts for <unk>
        unigram.update(updatedTokens) 
        for i in range(1, len(updatedTokens)):
            bigram[(updatedTokens[i-1], updatedTokens[i])] += 1
            
    return vocab, unigram, bigram

"""
------------------------------------Good Turing Smoothing------------------------------------
"""
def prevNc(prev, bigram_counts, vocab):
    Nc = Counter()
    tokensTot = 0
    seenVocab = 0
    
    for w in vocab:
        c = bigram_counts.get((prev, w), 0)
        if c > 0:
            Nc[c] += 1
            seenVocab += 1
            tokensTot += c

    return Nc, tokensTot, seenVocab

def goodTuring(prev, word, unigram_counts, bigram_counts, vocab):
    V = len(vocab)
    previousCount = unigram_counts.get(prev, 0)

    # (C(prev)=0) cannot use good turing smoothing 
    if previousCount == 0:
        # use unigram probability instead
        allWords = sum(unigram_counts.values())
        if allWords > 0:
            wordCount  = unigram_counts.get(word, 0)
            return max((wordCount/ allWords), tinyNum)
        return 1.0 / V

    Nc, tokensTot, seenVocab = prevNc(prev, bigram_counts, vocab)
    
    total = previousCount
    N0 = V - seenVocab
    N1 = Nc.get(1, 0) # how many words with 1 count after prev
    c = bigram_counts.get((prev, word), 0)

    # if bigram (prev, word) is unseen
    if c == 0:
        if total == 0 or N0 <= 0:
            return tinyNum
        
        probMass = 0.0

        if N1 > 0:
            # probability of the unseen mass  = N1 / C(prev)
            probMass = N1 / total

        unseenTypeProbs = 0.0

        if N0 > 0:
            unseenTypeProbs = probMass / N0

        return max(unseenTypeProbs, tinyNum)

    # getting c*
    Nc_c = Nc.get(c, 0)
    Nc_cp1 = Nc.get(c + 1, 0)

    if Nc_c > 0 and Nc_cp1 > 0:
        c_star = (c + 1) * (Nc_cp1 / Nc_c)
    else:
        c_star = c

    if total > 0:
        p = c_star / total
    else:
        p = tinyNum

    return max(p, tinyNum)

"""
------------------------------------Good Turing Smoothing End------------------------------------
"""

def logprob(sentence, vocab, uni_counts, bi_counts):
    tokens = tokenize(sentence)
    updatedTokens = [w if w in vocab else "<UNK>" for w in tokens]
    V = len(vocab)
    logp = 0.0
    for i in range(1, len(updatedTokens)):
        prev, w = updatedTokens[i-1], updatedTokens[i]
        p = goodTuring(prev, w, uni_counts, bi_counts, vocab)
        logp += math.log(p)
    return logp

def predict(line, models):
    # my models structure is models: {lang: (vocab, uni_counts, bi_counts)}
    scores = {}
    for lang, (vocab, uni, bi) in models.items():
        scores[lang] = logprob(line, vocab, uni, bi)
    best = max(scores, key=scores.get)
    return best, scores

def load_lines(path):
    with open(path, encoding="utf8") as f:
        return [makeLower(l.rstrip("\n")) for l in f]

def write_predictions(predictions, out_path):
    with open(out_path, "w", encoding="utf8") as f:
        f.writelines(predictions)

def evaluate_predictions(predictedLines, solutionLines):
    sol = load_lines(solutionLines)
    correct = 0
    total = min(len(predictedLines), len(sol))
    for pred, g in zip(predictedLines, sol):
        if pred.strip() == g.strip():
            correct += 1
    return correct, total

# I added in like the special quotation marks that i know french has but i cant hit 100% accuracy so there might be some other characters that i missed
# load data to train 
en_text = load_lines("../Data/Input/LangId.train.English")
fr_text  = load_lines("../Data/Input/LangId.train.French")
it_text  = load_lines("../Data/Input/LangId.train.Italian")

en_model = bigram(en_text)
fr_model = bigram(fr_text)
it_model = bigram(it_text)

models = {
    "English": en_model,
    "French": fr_model,
    "Italian": it_model
}

with open("../Data/Validation/LangId.test", encoding="utf8") as f:
    test_lines = [l.rstrip("\n") for l in f]

output_lines = []

for i, line in enumerate(test_lines, 1): #make sure it looks the same as the validation solution file 
    line_id = str(i)
    sentence = line.lower().strip()           
    pred, _ = predict(sentence, models)
    output_lines.append(f"{line_id} {pred}\n")

with open("../Data/Output/wordLangId.out", "w", encoding="utf8") as f:
    f.writelines(output_lines)

with open("../Data/Validation/labels.sol", encoding="utf8") as f:
    sol = f.readlines()

correct = 0
total = len(sol)

for mine, sol in zip(output_lines, sol):
    if mine.strip() == sol.strip():
        correct += 1

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total:.2%}")




Correct: 299/300
Accuracy: 99.67%
